In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb
from pathlib import Path

def _sql_literal(s: str) -> str:
    return s.replace("'", "''")

duck = connect_to_postgres_via_duckdb()

credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [2]:
key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

In [3]:
duck.sql("select 'TRUE' == 'TRUE'")

┌───────────────────┐
│ ('TRUE' = 'TRUE') │
│      boolean      │
├───────────────────┤
│ true              │
└───────────────────┘

In [3]:
duck.sql(
    """
    create or replace table pg.bas_firms.easybill_clean as 
    select 
        Kundennummer as easybill_id,
        Firma as firm_name,
        Address as address,
        "Zusatz 1" as address_complement1,
        "Zusatz 2" as address_complement2,
        merge == 'TRUE' as merge,
        destination as merge_mother,
        "simple collection\n-> Jeder Account erhält seine eigene Rechnung" == 'TRUE' as simple_collection,
        "accounting collection\n-> Mutterkonzern erhält alle Rechnungen" == 'TRUE' as accounting_collection,
        mother,
        "basic_care_billing_on_mother\n-> Anzuklicken, wenn alle die eigene Rechnung bekommen sollen, AUSSER für Grundbetreuung" == 'TRUE' as basic_care_simple_collection,
        "simple collection without mother" as headless_simple_collection_name
    from read_gsheet('1jX7xlXU4suEzmnRQn_m86ZmB7eKgONfYdYD4e8BYW4U', sheet='collection_or_duplicate', all_varchar=true)
    """
)

In [21]:
duck.sql(
    """
    select mother_client_id from pg.bas_firms.full_basic_care group by 1 having count(*) = 1
    """
)

┌──────────────────┐
│ mother_client_id │
│     varchar      │
├──────────────────┤
│ 101000048        │
│ 101000056        │
│ 103000022        │
│ 104000011        │
│ 104000012        │
│ 104000028        │
│ 105000007        │
│ 105010079        │
│ 106020016        │
│ 106020034        │
│     ·            │
│     ·            │
│     ·            │
│ 130001573        │
│ 130001596        │
│ 130001775        │
│ 130002143        │
│ 130002235        │
│ 130002296        │
│ 130002297        │
│ 130002313        │
│ 130002323        │
│ 130002394        │
├──────────────────┤
│     786 rows     │
│    (20 shown)    │
└──────────────────┘

In [4]:
duck.sql(
    """
    select * 
    from pg.bas_firms.easybill_clean
    where easybill_id in (    select mother_client_id from pg.bas_firms.full_basic_care group by 1 having count(*) = 1)
    order by 
    """
)

ParserException: Parser Error: syntax error at end of input